<a href="https://colab.research.google.com/github/zunaidalam404/Nano-GPT_0111/blob/main/clone_data_generate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Clone your GitHub repository
!git clone https://github.com/zunaidalam404/Nano-GPT_0111.git

# Enter project directory
%cd Nano-GPT_0111

# Install Hugging Face library
!pip install -q huggingface_hub

Cloning into 'Nano-GPT_0111'...
remote: Enumerating objects: 35, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 35 (delta 10), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (35/35), 509.01 KiB | 4.04 MiB/s, done.
Resolving deltas: 100% (10/10), done.
/content/Nano-GPT_0111


In [2]:
!pip install -q huggingface_hub

In [3]:
import torch
import huggingface_hub

print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
print("Libraries installed successfully!")

PyTorch: 2.14.1+cpu
CUDA Available: False
Libraries installed successfully!


In [4]:
from huggingface_hub import hf_hub_download

REPO_ID = "zunaidalam/my-custom-gpt"

# Download trained GPT weights
model_path = hf_hub_download(
    repo_id=REPO_ID,
    filename="gpt_model[1].pth"
)

# Download tokenizer
tokenizer_path = hf_hub_download(
    repo_id=REPO_ID,
    filename="tokenizer[1].json"
)

print("Model downloaded:", model_path)
print("Tokenizer downloaded:", tokenizer_path)
print("Both files downloaded successfully!")

gpt_model[1].pth: reconstructing file:   0%|          |  0.00B / 52.7MB            

gpt_model[1].pth: downloading bytes:           |  0.00B            

tokenizer%5B1%5D.json:   0%|          | 0.00/326 [00:00<?, ?B/s]

Model downloaded: /root/.cache/huggingface/hub/models--zunaidalam--my-custom-gpt/snapshots/2df1296564eca3ce7cdfbe67bbc4e6ecad4810c9/gpt_model[1].pth
Tokenizer downloaded: /root/.cache/huggingface/hub/models--zunaidalam--my-custom-gpt/snapshots/2df1296564eca3ce7cdfbe67bbc4e6ecad4810c9/tokenizer[1].json
Both files downloaded successfully!


In [5]:
import torch
from model import GPTLanguageModel

# Use CPU
device = "cpu"

# Create GPT architecture
model = GPTLanguageModel().to(device)

# Load trained weights
checkpoint = torch.load(
    model_path,
    map_location=device,
    weights_only=True
)

# Load weights into model
model.load_state_dict(checkpoint)

# Set evaluation mode
model.eval()

print("GPT Model Loaded Successfully!")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())

print("Total Parameters:", total_params)

GPT Model Loaded Successfully!
Total Parameters: 10788929


In [6]:
from tokenizer import CharTokenizer
import torch

# Load tokenizer
tokenizer = CharTokenizer(tokenizer_path)

# Give prompt
prompt = "ROMEO:"

# Convert text into tokens
context = torch.tensor(
    [tokenizer.encode(prompt)],
    dtype=torch.long,
    device=device
)

# Generate text
with torch.no_grad():
    generated = model.generate(
        context,
        max_new_tokens=300
    )

# Convert tokens into text
response = tokenizer.decode(
    generated[0].tolist()
)

print("GPT Response:\n")
print(response)

GPT Response:

ROMEO:
Good good my lord;
And fellow here hath strangling thereof:
Take sends mark'd them from the eagle-breath,
And then feasting more of their true births:
Long knightning lives wail by a diaper's womb
And stood, as thou hear'st by this rain bard's heart.
The day is valour'd too long; if I love take so,


In [7]:
%cd /content/Nano-GPT_0111
!git pull
!pip install -r requirements.txt

/content/Nano-GPT_0111
remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 3 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 1.34 KiB | 1.34 MiB/s, done.
From https://github.com/zunaidalam404/Nano-GPT_0111
   cd4d645..f1992ef  main       -> origin/main
Updating cd4d645..f1992ef
Fast-forward
 main.py | 59 +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
 1 file changed, 59 insertions(+)
 create mode 100644 main.py


In [11]:
# !cat main.py

In [12]:
# !cat inference.py!cat inference.py

In [13]:
from inference import generate_response

response = generate_response(
    "ROMEO:",
    max_new_tokens=100
)

print(response)

ROMEO:
Juliet heart! more weep your sorrow needs
To weep dreamt, and still betray something
That monary cr


In [14]:
import subprocess
import time
import requests

server = subprocess.Popen(
    ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
)

time.sleep(5)

response = requests.get("http://127.0.0.1:8000/")

print(response.status_code)
print(response.json())

200
{'message': 'My Custom GPT API is running!'}


In [15]:
import requests

url = "http://127.0.0.1:8000/chat"

data = {
    "message": "ROMEO:"
}

response = requests.post(url, json=data)

print("Status:", response.status_code)
print("GPT Response:", response.json())

Status: 200
GPT Response: {'question': 'ROMEO:', 'answer': 'ROMEO:\nThe Lord Northumberland now\nEven there sentence on them traitors prophesy.\n\nJOHN OF GAUNT:\nThe king is a citor hath provoked the law:\nAnd when the Capulets did stay the people\nWill war with his urnap'}


In [16]:
!pip install -q streamlit requests

In [17]:
%%writefile streamlit_app.py

import streamlit as st
import requests


# Page Configuration
st.set_page_config(
    page_title="My Custom GPT",
    page_icon="🤖",
    layout="centered"
)


# FastAPI Backend URL
API_URL = "http://127.0.0.1:8000/chat"


# Initialize Session State
if "messages" not in st.session_state:
    st.session_state.messages = []


# Heading
st.title("🤖 My Custom GPT")

st.caption("Powered by My Own Trained GPT Model")


# Sidebar
with st.sidebar:

    st.title("Settings")

    if st.button("🗑️ Clear Chat"):

        st.session_state.messages = []

        st.rerun()

    st.info("Model: Nano-GPT")

    st.success("Backend: FastAPI")


# Display Chat History
for message in st.session_state.messages:

    with st.chat_message(message["role"]):

        st.markdown(message["content"])


# Chat Input
prompt = st.chat_input("Message My Custom GPT...")


if prompt:

    # Display User Message
    st.session_state.messages.append({
        "role": "user",
        "content": prompt
    })

    with st.chat_message("user"):
        st.markdown(prompt)


    # Generate AI Response
    with st.chat_message("assistant"):

        with st.spinner("GPT is thinking..."):

            try:

                response = requests.post(
                    API_URL,
                    json={"message": prompt},
                    timeout=180
                )

                response.raise_for_status()

                data = response.json()

                answer = data["answer"]

                st.markdown(answer)

                st.session_state.messages.append({
                    "role": "assistant",
                    "content": answer
                })

            except requests.exceptions.RequestException as e:

                st.error(f"Backend Error: {e}")

Writing streamlit_app.py


In [40]:
!fuser -k 8502/tcp

8502/tcp:            26614


In [41]:
import subprocess
import time

streamlit_server = subprocess.Popen(
    [
        "streamlit",
        "run",
        "streamlit_app.py",
        "--server.port", "8502",
        "--server.address", "0.0.0.0",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false"
    ],
    stdout=open("streamlit_8502.log", "w"),
    stderr=subprocess.STDOUT
)

time.sleep(8)

print("Streamlit restarted successfully!")

Streamlit restarted successfully!


In [42]:
import requests

r = requests.get("http://127.0.0.1:8502/_stcore/health")

print("Status:", r.status_code)
print("Response:", r.text)

Status: 200
Response: ok


In [43]:
from google.colab.output import eval_js

url = eval_js("google.colab.kernel.proxyPort(8502)")

print("MY NANO GPT:")
print(url)

MY NANO GPT:
https://8502-m-s-use1b10-imr3vpa75576-b.us-east1.prod.r.colab.dev
